# 1. Synthetic Data Generation
This notebook documents the process of expanding the original dataset from 1,000 records to 12,000 records using a bootstrap-and-jitter method. This method preserves statistical correlations, marginal distributions, and range limits of the original features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load original dataset
df_orig = pd.read_csv('../old_ML_Project/student_performance.csv')
print("Original dataset size:", df_orig.shape)

### Jittering and Sampling Parameters
We use a bootstrap technique with small continuous noise (jitter) for numerical columns and a minor mutation rate for categorical columns. This ensures all 11,000 generated records are unique while maintaining correlations.

In [ ]:
num_bounds = {
    "age": (17, 30, True),
    "semester": (1, 8, True),
    "study_hours_per_week": (0.0, 50.0, False),
    "attendance_percentage": (0.0, 100.0, False),
    "assignment_average": (0.0, 100.0, False),
    "midterm_score": (0.0, 100.0, False),
    "previous_gpa": (0.0, 4.0, False),
    "extracurricular_hours_per_week": (0.0, 30.0, False),
    "absences": (0, 60, True),
    "final_score": (0.0, 100.0, False)
}
categorical_cols = ["gender", "department", "internet_access", "extra_academic_support", "part_time_job"]

np.random.seed(42)
num_samples = 11000
boot_indices = np.random.choice(len(df_orig), size=num_samples, replace=True)
df_synth = df_orig.iloc[boot_indices].copy().reset_index(drop=True)

# Apply jitter
for col, (vmin, vmax, is_int) in num_bounds.items():
    col_std = df_orig[col].std()
    noise = np.random.normal(0, col_std * 0.08, size=num_samples)
    if is_int:
        df_synth[col] = (df_synth[col] + noise).round().astype(int)
    else:
        df_synth[col] = df_synth[col] + noise
    df_synth[col] = df_synth[col].clip(vmin, vmax)

# Mutate categoricals slightly
for col in categorical_cols:
    uniq_vals = df_orig[col].unique()
    mutate_mask = np.random.rand(num_samples) < 0.03
    if mutate_mask.any():
        df_synth.loc[mutate_mask, col] = np.random.choice(uniq_vals, size=mutate_mask.sum())

df_synth["student_id"] = [f"STU{i}" for i in range(2001, 2001 + num_samples)]
df_synth["data_origin"] = "Synthetic"
df_orig["data_origin"] = "Original"
df_final = pd.concat([df_orig, df_synth], ignore_index=True)
print("Final merged dataset size:", df_final.shape)

### Statistical Comparisons

In [ ]:
print("Class balance in Original dataset:")
print(df_orig['at_risk'].value_counts(normalize=True))
print("\nClass balance in Synthetic dataset:")
print(df_synth['at_risk'].value_counts(normalize=True))

print("\nCorrelation of numerical features with at_risk (Original vs Synthetic):")
orig_corr = df_orig[list(num_bounds.keys()) + ['at_risk']].corr()['at_risk']
synth_corr = df_synth[list(num_bounds.keys()) + ['at_risk']].corr()['at_risk']
print(pd.DataFrame({'Original': orig_corr, 'Synthetic': synth_corr}))